# Notebook 3: RLHF with DPO (Direct Preference Optimization)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BalaAnbalagan/modern-ai-unsloth/blob/main/colab3_rlhf.ipynb)

**Author**: Balamuralikrishnan Anbalagan  
**Objective**: Demonstrate preference-based training using DPO on Anthropic HH-RLHF dataset

---

## Overview
This notebook demonstrates **Direct Preference Optimization (DPO)**, a simpler alternative to traditional RLHF. We'll:
- Use Anthropic's HH-RLHF dataset with human preference labels
- Train with DPO to align model with preferred responses
- Track reward progression and preference accuracy
- Compare chosen vs rejected responses

## 1. Installation & Setup

In [ ]:
%%capture
# Install Unsloth and dependencies
# What's happening: Installing the Unsloth library with DPO support
# Key libraries for RLHF/DPO:
#   - unsloth: Provides optimized DPO training (2x faster than standard)
#   - trl: Contains DPOTrainer for preference-based learning
#   - peft: Implements LoRA for parameter-efficient preference tuning
#   - bitsandbytes: Enables memory-efficient 8-bit optimizers for DPO
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# Verify GPU availability
# What's happening: Checking GPU specs for DPO training
# DPO note: Requires more memory than SFT because it processes both chosen and rejected responses
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"BF16 Support: {torch.cuda.is_bf16_supported()}")

## 2. Load Anthropic HH-RLHF Dataset

This dataset contains 161k conversations with human preference labels:
- **chosen**: Preferred response (helpful, harmless, honest)
- **rejected**: Less preferred response

In [ ]:
from datasets import load_dataset

# Load subset for quick training (1000 samples)
# What's happening: Loading Anthropic's HH-RLHF (Helpful and Harmless) dataset
# Dataset structure:
#   - Each sample has a conversation with TWO possible responses
#   - "chosen": Response rated as more helpful/harmless by humans
#   - "rejected": Response rated as less helpful/harmless
# How DPO uses this:
#   The model learns to increase probability of "chosen" responses
#   and decrease probability of "rejected" responses
print("Loading Anthropic HH-RLHF dataset...")
dataset = load_dataset("Anthropic/hh-rlhf", split="train[:1000]")

print(f"\n✓ Dataset loaded: {len(dataset)} samples")
print(f"  Fields: {dataset.column_names}")

# Show example
print("\n" + "="*80)
print("SAMPLE PREFERENCE PAIR")
print("="*80)
print("\nChosen (Preferred):")
print("-" * 80)
print(dataset[0]['chosen'][:300])
print("\nRejected (Less Preferred):")
print("-" * 80)
print(dataset[0]['rejected'][:300])
print("="*80)

## 3. Load Model with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

# Configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Load model and tokenizer
# What's happening: Loading the same SmolLM2-135M with 4-bit quantization
# Unsloth optimizations for DPO:
#   1. Efficient dual forward passes (for chosen AND rejected responses)
#   2. Shared computation between reference and policy models
#   3. Memory-efficient KL divergence calculation
#   4. Optimized gradient computation for preference loss
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/smollm2-135m",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add padding token if needed
# Why: DPO requires batch processing of chosen/rejected pairs
# Padding ensures all sequences in batch have same length
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✓ Model loaded: {model.config._name_or_path}")
print(f"✓ Total parameters: {model.num_parameters():,}")

## 4. Apply LoRA for DPO Training

Use higher rank (64) for DPO as preference learning benefits from more expressiveness

In [ ]:
# Apply LoRA with higher rank for DPO
# What's happening: Adding LoRA adapters with rank=64 (higher than colab2's rank=8)
# Why higher rank for DPO:
#   - Preference learning is more nuanced than simple task adaptation
#   - Model needs to learn subtle differences between chosen/rejected responses
#   - Higher rank = more expressive adapters = better preference capture
#   - Rank 64 is a sweet spot: expressive enough without too many parameters
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,  # Higher rank for preference learning (vs 8 for simple LoRA)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,  # Typically set to match rank for DPO
    lora_dropout = 0,  # No dropout for DPO (helps stability)
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = False,
)

# Calculate trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = model.num_parameters()
print(f"\n✓ LoRA Applied for DPO Training")
print(f"  Trainable params: {trainable_params:,}")
print(f"  Total params: {total_params:,}")
print(f"  Trainable %: {trainable_params/total_params*100:.2f}%")
print(f"  LoRA Rank: 64")

## 5. Prepare Dataset for DPO

DPO requires specific format with prompt, chosen, and rejected fields

In [ ]:
# Function to extract prompt from conversation
def extract_prompt(text):
    """Extract the prompt (human input) from the conversation."""
    if "Human:" in text:
        # Get everything up to and including first human message
        parts = text.split("Assistant:", 1)
        if len(parts) > 0:
            return parts[0].strip()
    return text.split("\n\n")[0]  # Fallback: first paragraph

# Format dataset for DPO
# What's happening: Restructuring the dataset into prompt/chosen/rejected format
# Why this matters:
#   - DPOTrainer expects three fields: prompt, chosen, rejected
#   - Original HH-RLHF has full conversations in chosen/rejected
#   - We need to extract the common prompt (human input)
#   - Then separate the two different responses (assistant outputs)
def format_for_dpo(example):
    prompt = extract_prompt(example['chosen'])
    
    # Extract response parts
    chosen_response = example['chosen'].replace(prompt, "").strip()
    rejected_response = example['rejected'].replace(prompt, "").strip()
    
    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response,
    }

# Apply formatting
print("Formatting dataset for DPO...")
dataset = dataset.map(format_for_dpo)

print("\n✓ Dataset formatted for DPO")
print(f"  Fields: {dataset.column_names}")
print("\nSample formatted example:")
print(f"  Prompt: {dataset[5]['prompt'][:100]}...")
print(f"  Chosen: {dataset[5]['chosen'][:100]}...")
print(f"  Rejected: {dataset[5]['rejected'][:100]}...")

## 6. Configure DPO Training

In [ ]:
from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
from unsloth import PatchDPOTrainer
import os

# MUST call this before using DPOTrainer with Unsloth
# What this does: Patches DPOTrainer to use Unsloth's optimized kernels
# Without this patch, DPO would be much slower and use more memory
PatchDPOTrainer()

# Create checkpoint directory
output_dir = "./checkpoints/colab3"
os.makedirs(output_dir, exist_ok=True)

# DPO Configuration
# What's happening: Setting up DPO-specific training parameters
# Key DPO concepts:
#   - Beta (temperature): Controls how much model can deviate from reference
#     * Low beta (0.01): Model stays very close to original (conservative)
#     * High beta (1.0): Model can change more dramatically (risky)
#     * We use 0.1 as a balanced middle ground
#   - Reference model: The original model before DPO (used for KL divergence)
#     * Unsloth efficiently handles this internally (no extra memory needed!)
#   - Bradley-Terry model: Mathematical framework for pairwise preferences
training_args = DPOConfig(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    num_train_epochs = 1,
    max_steps = 100,
    learning_rate = 5e-5,  # Lower than SFT to prevent catastrophic forgetting
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 42,
    output_dir = output_dir,
    save_strategy = "steps",
    save_steps = 50,
    report_to = "none",
    # DPO-specific parameters
    beta = 0.1,  # DPO temperature parameter (controls KL penalty)
    max_length = 1024,
    max_prompt_length = 512,
)

print("✓ DPO Training configuration:")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Max steps: {training_args.max_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Beta (temperature): {training_args.beta}")
print(f"  Max sequence length: {training_args.max_length}")

## 7. Initialize DPO Trainer & Start Training

In [ ]:
# Initialize DPO Trainer
# What's happening: Creating a specialized trainer for preference-based learning
# How DPO works (simplified):
#   1. For each training sample, the model processes BOTH chosen and rejected responses
#   2. Calculate log probabilities for both responses
#   3. Calculate log probabilities from reference model (original frozen model)
#   4. Compute DPO loss:
#      - Increase probability of chosen response
#      - Decrease probability of rejected response
#      - Penalize deviation from reference model (using beta parameter)
#   5. Update only the LoRA adapters (base model stays frozen)
# Unsloth optimizations:
#   - Shares computation between policy and reference model evaluations
#   - Fused kernels for dual forward passes
#   - Memory-efficient handling of reference model (no extra copy needed!)
dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,  # Unsloth handles reference model internally (smart!)
    args = training_args,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

print("\n" + "="*80)
print("STARTING DPO TRAINING - Preference-Based Learning")
print("="*80)

# Monitor GPU memory
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    print(f"\nGPU Memory before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Train with DPO
trainer_stats = dpo_trainer.train()

# Monitor GPU memory after training
if torch.cuda.is_available():
    print(f"\nGPU Memory after training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"Peak GPU Memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

print("\n" + "="*80)
print("DPO TRAINING COMPLETED")
print("="*80)

## 8. Analyze Training Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract training logs
logs = dpo_trainer.state.log_history
train_logs = [log for log in logs if 'loss' in log]

# Create DataFrame
df = pd.DataFrame(train_logs)
print("\nDPO Training Statistics:")
if 'rewards/chosen' in df.columns:
    print(df[['step', 'loss', 'rewards/chosen', 'rewards/rejected']].to_string(index=False))
else:
    print(df[['step', 'loss', 'learning_rate']].to_string(index=False))

# Plot loss and rewards
if len(df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curve
    axes[0].plot(df['step'], df['loss'], marker='o', linewidth=2, color='purple')
    axes[0].set_xlabel('Training Step', fontsize=12)
    axes[0].set_ylabel('DPO Loss', fontsize=12)
    axes[0].set_title('DPO Loss Curve', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Reward progression (if available)
    if 'rewards/chosen' in df.columns:
        axes[1].plot(df['step'], df['rewards/chosen'], marker='o', 
                    linewidth=2, color='green', label='Chosen')
        axes[1].plot(df['step'], df['rewards/rejected'], marker='s', 
                    linewidth=2, color='red', label='Rejected')
        axes[1].set_xlabel('Training Step', fontsize=12)
        axes[1].set_ylabel('Reward', fontsize=12)
        axes[1].set_title('Reward Progression', fontsize=14, fontweight='bold')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].plot(df['step'], df['learning_rate'], marker='o', linewidth=2)
        axes[1].set_xlabel('Training Step', fontsize=12)
        axes[1].set_ylabel('Learning Rate', fontsize=12)
        axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{output_dir}/dpo_metrics.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n✓ DPO metrics saved to {output_dir}/dpo_metrics.png")

# Print final statistics
print(f"\nFinal DPO Statistics:")
print(f"  Total steps: {dpo_trainer.state.global_step}")
print(f"  Final loss: {df['loss'].iloc[-1]:.4f}")
if 'rewards/chosen' in df.columns:
    print(f"  Final chosen reward: {df['rewards/chosen'].iloc[-1]:.4f}")
    print(f"  Final rejected reward: {df['rewards/rejected'].iloc[-1]:.4f}")
    print(f"  Reward margin: {df['rewards/chosen'].iloc[-1] - df['rewards/rejected'].iloc[-1]:.4f}")

## 9. Test Preference Alignment

In [ ]:
# Enable fast inference mode
# What's happening: Optimizing model for generation after DPO training
# Unsloth inference optimizations for DPO models:
#   1. Merges LoRA adapters into base weights (faster than separate computation)
#   2. Removes reference model tracking (not needed for inference)
#   3. Enables efficient KV-cache for autoregressive generation
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "Human: What is the best way to learn programming?\n\nAssistant:",
    "Human: How can I be more productive?\n\nAssistant:",
    "Human: What should I do if I'm feeling stressed?\n\nAssistant:",
]

print("\n" + "="*80)
print("TESTING PREFERENCE-ALIGNED RESPONSES")
print("="*80)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n--- Test {i} ---")
    print(f"Prompt: {prompt.split('Assistant:')[0]}")
    print("\nGenerated Response:")
    print("-" * 80)
    
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    # Generate response
    # What to expect after DPO:
    #   - More helpful, harmless, and honest responses
    #   - Better aligned with human preferences
    #   - Reduced toxic or biased outputs
    #   - More natural conversation flow
    outputs = model.generate(
        **inputs,
        max_new_tokens = 150,
        temperature = 0.7,
        top_p = 0.9,
        do_sample = True,
        use_cache = True,
    )
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the assistant's response
    if "Assistant:" in generated_text:
        response = generated_text.split("Assistant:")[-1].strip()
        print(response)
    else:
        print(generated_text)
    print("-" * 80)

## 10. Save Model Checkpoints

In [ ]:
# Save DPO-trained LoRA adapter
# What's happening: Saving the preference-aligned adapter weights
# DPO adapter uses:
#   - Can be loaded on top of base model for preference-aligned responses
#   - Much smaller than full model (only stores the learned preferences)
#   - Can be combined with other adapters for multi-task models
lora_path = f"{output_dir}/dpo_adapter"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"✓ DPO adapter saved to {lora_path}")

# Save merged model
# What's happening: Merging LoRA weights into base model for easier deployment
# merged_16bit: Converts from 4-bit back to 16-bit with adapters merged
# Why: Easier to deploy, no need to load separate adapter files
merged_path = f"{output_dir}/merged_16bit"
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"✓ Merged model saved to {merged_path}")

print("\n✓ All checkpoints saved successfully!")

## 11. Summary & Observations

### Key Results:
- **Training Method**: Direct Preference Optimization (DPO)
- **Model**: SmolLM2-135M (135M parameters)
- **Dataset**: Anthropic HH-RLHF (1000 preference pairs)
- **Training Steps**: 100 steps
- **GPU**: Google Colab T4 (12GB VRAM)

### What is DPO?
**Direct Preference Optimization** simplifies RLHF by:
1. Eliminating the separate reward model training phase
2. Directly optimizing policy to prefer chosen responses over rejected ones
3. Using a simple loss function based on Bradley-Terry preference model
4. Maintaining reference to original model to prevent collapse

### DPO vs Traditional RLHF:
| Aspect | Traditional RLHF | DPO |
|--------|------------------|-----|
| Reward Model | Separate training required | Not needed |
| Complexity | High (3 stages) | Low (1 stage) |
| Stability | Can be unstable | More stable |
| Memory | Requires 2 models | Single model |
| Training Time | Longer | Faster |

### Observations:
1. **Reward Margin**: Chosen responses should have higher rewards than rejected
2. **Loss Convergence**: DPO loss decreases as model learns preferences
3. **Response Quality**: Model generates more helpful, harmless responses
4. **Efficiency**: Single-stage training is simpler than full RLHF

### Use Cases for DPO:
- ✓ Aligning chatbots with human preferences
- ✓ Teaching helpfulness, harmlessness, honesty (HHH)
- ✓ Reducing toxic or biased outputs
- ✓ Improving response quality for specific domains
- ✓ Fine-tuning with preference feedback

### Key Hyperparameters:
- **Beta (0.1)**: Controls how much to deviate from reference model (lower = more conservative)
- **Learning Rate (5e-5)**: Lower than SFT to prevent overfitting
- **LoRA Rank (64)**: Higher rank helps capture preference nuances

---

**Next**: See [colab4_grpo_reasoning.ipynb](colab4_grpo_reasoning.ipynb) for reasoning training with GRPO!